# Pipeline Bottleneck Experiment — Final Thesis Version

**Purpose:** Definitively measure where latency goes in the S0 baseline pipeline (router → analyze_data → extract_chart_config → justify_chart_type → render).

**Design principle:** The original `TimedAgent` pipeline execution path is preserved exactly — `TokenTrackedAgent` only adds reliable token capture via `create_with_completion`. All stage timing still comes from `TimedAgent`.

### Experiments
| # | What | N runs | Why |
|---|------|--------|-----|
| 1 | Single run — full stage breakdown | 1 | See per-step time + tokens |
| 2 | Repeated runs — averages | 5 | Smooth per-stage noise |
| 3 | Model comparison: o4-mini vs gpt-4o-mini | 3 each | Isolate reasoning overhead |
| 4 | Prompt size scaling | 2 per size | Does token count drive latency? |
| 5 | Variance + reasoning-token correlation | 10 | Explains cold/warm gap to advisor |

### Output
- Per-run CSV with all timing and token columns
- Summary statistics: mean, median, std, min, max, P90, P95, CV
- Stage breakdown table: avg latency and avg token usage per stage

> **Thesis note:** No caching is active. Phoenix tracing is optional — if Phoenix is not running the notebook falls back to a no-op tracer and runs identically.

In [1]:
# ═══════════════════════════════════════════════════════════════════
# Cell 1 — Setup (run once before any experiment)
# ═══════════════════════════════════════════════════════════════════
import os, sys, time, json, copy, statistics, csv
from collections import OrderedDict
from pathlib import Path

import instructor
from openai import OpenAI
from dotenv import load_dotenv
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from pydantic import ValidationError
from opentelemetry.trace import StatusCode

# ── Path setup ────────────────────────────────────────────────────────
HERE = Path("/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/FINAL FOLDER")
REPO = HERE.parent
for p in (str(HERE), str(REPO)):
    if p not in sys.path:
        sys.path.insert(0, p)

# ── Load .env ─────────────────────────────────────────────────────────
_ENV = Path("/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/.env")
load_dotenv(_ENV, override=True)
if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(f"OPENAI_API_KEY missing — check .env at {_ENV}")
print(f"✓ .env loaded  |  API key set: {bool(os.getenv('OPENAI_API_KEY'))}")

# ── Imports from project ──────────────────────────────────────────────
from retrieve_data import retrieve_data
from generate_visualization_timed import TimedAgent
from prompts.default import (
    DATA_ANALYSIS_PROMPT,
    CHART_CONFIGURATION_PROMPT,
    CREATE_CHART_TYPE_JUSTIFICATION_PROMPT,
    SYSTEM_PROMPT,
)
from response_models.default import (
    DataAnalysis, VisualizationConfig,
    ChartTypeJustification, ChartType,
)
try:
    from visualization_from_template import generate_from_template
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    RENDER_AVAILABLE = True
except ImportError:
    RENDER_AVAILABLE = False
    print("⚠  visualization_from_template not found — render stage will be skipped")

# ── Phoenix (optional) — falls back to no-op tracer ───────────────────
class _NoOpSpan:
    def set_input(self, value=None, **kw): pass
    def set_output(self, value=None, **kw): pass
    def set_status(self, *a, **kw): pass
    def __enter__(self): return self
    def __exit__(self, *a): pass

class _NoOpTracer:
    def start_as_current_span(self, *a, **kw): return _NoOpSpan()
    def tool(self, fn): return fn
    def chain(self, fn): return fn

try:
    from init_phoenix import init_phoenix
    inst_client, raw_client, tracer = init_phoenix()
    PHOENIX_ACTIVE = True
    print("✓ Phoenix tracing active")
except Exception as _e:
    print(f"Phoenix not available ({_e}) — using no-op tracer")
    raw_client  = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    inst_client = instructor.from_openai(raw_client)
    tracer      = _NoOpTracer()
    PHOENIX_ACTIVE = False

# ── Fixed test data ───────────────────────────────────────────────────
QUESTION = (
    "Wieviel Umsatz hatte Teckentrup in den Jahren 2021 bis 2024 "
    "im Segment JVA? Provide a detailed analysis of the data based "
    "on total earnings and provide a comprehensive visualization "
    "supporting your analysis."
)
MD_TABLE = retrieve_data(None, type="test")
MESSAGES = [
    {"role": "user",   "content": QUESTION},
    {"role": "system", "content": f"The data is: {MD_TABLE}"},
]

MODEL = "o4-mini"  # primary model
REASONING_MODELS = {"o1", "o1-mini", "o3", "o3-mini", "o4-mini"}

char_count = sum(len(m["content"]) for m in MESSAGES)
print(f"\n✓ Data loaded. Prompt size: {char_count} chars (~{char_count // 4} tokens)")
print(f"Primary model: {MODEL}  |  Phoenix active: {PHOENIX_ACTIVE}")
print(f"Render available: {RENDER_AVAILABLE}")

# ── Global collector for CSV export ───────────────────────────────────
ALL_RESULTS = []   # all experiment rows accumulate here

✓ .env loaded  |  API key set: True
🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: tracing-agent
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

✓ Phoenix tracing active

✓ Data loaded. Prompt size: 3625 chars (~906 tokens)
Primary model: o4-mini  |  Phoenix active: True
Render available: True


In [2]:
# ═══════════════════════════════════════════════════════════════════
# Cell 2 — TokenTrackedAgent
# Extends TimedAgent with reliable token capture via create_with_completion.
# Pipeline execution path and all stage timings are IDENTICAL to TimedAgent.
# ═══════════════════════════════════════════════════════════════════

class TokenTrackedAgent(TimedAgent):
    """
    Subclass of TimedAgent that captures real token usage (including reasoning
    tokens from o4-mini) via instructor's create_with_completion.

    All pipeline stages and timing instrumentation are preserved from TimedAgent.
    The only change is that the three LLM stage methods call create_with_completion
    instead of create, so completion.usage is available and captured reliably.
    Router tokens are captured from response.usage in the overridden run_agent.

    Token data is available after each run in:
      agent.stage_tokens  —  {stage: {prompt, completion, reasoning}}
    """

    def __init__(self, client, tool_calling_client, tracer, model):
        self.stage_tokens = {}   # reset in start_main_span on each run
        super().__init__(client, tool_calling_client, tracer, model)

    @staticmethod
    def _tok(usage) -> dict:
        """Extract prompt / completion / reasoning tokens from an OpenAI usage object."""
        if usage is None:
            return {"prompt": 0, "completion": 0, "reasoning": 0}
        details  = getattr(usage, "completion_tokens_details", None)
        reasoning = getattr(details, "reasoning_tokens", 0) or 0 if details else 0
        return {
            "prompt":     getattr(usage, "prompt_tokens",     0) or 0,
            "completion": getattr(usage, "completion_tokens", 0) or 0,
            "reasoning":  reasoning,
        }

    def start_main_span(self, messages):
        self.stage_tokens = {}           # reset token store for each run
        return super().start_main_span(messages)

    # ── LLM stage overrides — same logic, adds create_with_completion ──

    def analyze_data(self, data: str, query: str) -> str:
        prompt = DATA_ANALYSIS_PROMPT.format(data=data, query=query)
        self.token_usage.setdefault("analyze_data", {})["prompt_chars"] = len(prompt)

        response, completion = self.client.chat.completions.create_with_completion(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            response_model=DataAnalysis,
        )
        tok = self._tok(completion.usage)
        self.stage_tokens["analyze_data"] = tok
        self.token_usage["analyze_data"].update({
            "prompt_tokens":     tok["prompt"],
            "completion_tokens": tok["completion"],
            "reasoning_tokens":  tok["reasoning"],
        })
        return response.analysis

    def extract_chart_config(self, data: str, analysis: str) -> dict:
        prompt = CHART_CONFIGURATION_PROMPT.format(data=data, analysis=analysis)
        self.token_usage.setdefault("extract_chart_config", {})["prompt_chars"] = len(prompt)

        response, completion = self.client.chat.completions.create_with_completion(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            response_model=VisualizationConfig,
        )
        tok = self._tok(completion.usage)
        self.stage_tokens["extract_chart_config"] = tok
        self.token_usage["extract_chart_config"].update({
            "prompt_tokens":     tok["prompt"],
            "completion_tokens": tok["completion"],
            "reasoning_tokens":  tok["reasoning"],
        })
        res = response.model_dump()
        res["charttype"] = res["charttype"].value
        return res

    @retry(
        retry=retry_if_exception_type(ValidationError),
        stop=stop_after_attempt(4),
        wait=wait_exponential(multiplier=1, min=4, max=10),
    )
    def justify_chart_type(self, analysis, data: str) -> str:
        charttypes = {ct.name for ct in ChartType}
        prompt = CREATE_CHART_TYPE_JUSTIFICATION_PROMPT.format(
            charttypes=charttypes, analysis=analysis, data=data
        )
        self.token_usage.setdefault("justify_chart_type", {})["prompt_chars"] = len(prompt)

        response, completion = self.client.chat.completions.create_with_completion(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            response_model=ChartTypeJustification,
        )
        tok = self._tok(completion.usage)
        self.stage_tokens["justify_chart_type"] = tok
        self.token_usage["justify_chart_type"].update({
            "prompt_tokens":     tok["prompt"],
            "completion_tokens": tok["completion"],
            "reasoning_tokens":  tok["reasoning"],
        })
        return response.chart_type.value

    # ── Router override — same logic as TimedAgent + token capture ─────

    def run_agent(self, messages):
        if isinstance(messages, str):
            messages = [{"role": "user", "content": messages}]

        if not any(isinstance(m, dict) and m.get("role") == "system" for m in messages):
            messages.append({"role": "system", "content": SYSTEM_PROMPT})

        while True:
            with self.tracer.start_as_current_span(
                "router_call", openinference_span_kind="chain"
            ) as span:
                span.set_input(value=messages)

                s = time.perf_counter()
                response = self.tool_calling_client.chat.completions.create(
                    model=self.model,
                    messages=messages,
                    tools=self.tools,
                )
                self.timings["router"] = time.perf_counter() - s

                # Capture router tokens from response.usage
                tok = self._tok(response.usage)
                self.stage_tokens["router"] = tok
                self.token_usage["router"] = {
                    "prompt_tokens":     tok["prompt"],
                    "completion_tokens": tok["completion"],
                    "reasoning_tokens":  tok["reasoning"],
                }

                messages.append(response.choices[0].message)
                tool_calls = response.choices[0].message.tool_calls
                span.set_status(StatusCode.OK)

                if tool_calls:
                    s = time.perf_counter()
                    messages = self.handle_tool_calls(tool_calls, messages)
                    self.timings["tool_execution_total"] = time.perf_counter() - s
                    span.set_output(value=tool_calls)
                    return messages[-1]
                else:
                    s = time.perf_counter()
                    resp_final = self.generate_final_response(messages)
                    self.timings["final_response"] = time.perf_counter() - s
                    span.set_output(value=resp_final)
                    return resp_final

    def get_all_stage_tokens(self) -> dict:
        """Return flat summary of all stage token counts after a run."""
        stages = ["router", "analyze_data", "extract_chart_config", "justify_chart_type"]
        totals = {"prompt": 0, "completion": 0, "reasoning": 0}
        per_stage = {}
        for s in stages:
            tok = self.stage_tokens.get(s, {"prompt": 0, "completion": 0, "reasoning": 0})
            per_stage[s] = tok
            for k in totals:
                totals[k] += tok[k]
        totals["total"] = totals["prompt"] + totals["completion"]
        return {"per_stage": per_stage, "total": totals}

print("✓ TokenTrackedAgent defined")

✓ TokenTrackedAgent defined


In [3]:
# ═══════════════════════════════════════════════════════════════════
# Cell 3 — Helper: run one full pipeline + render, return CSV row
# ═══════════════════════════════════════════════════════════════════

def run_one(run_num: int, model: str, experiment: str,
            messages=None, inst_c=None, raw_c=None, trc=None) -> dict:
    """
    Run one complete pipeline iteration (TimedAgent execution path + render).
    Returns a flat dict with all timing and token columns for CSV.

    Parameters
    ----------
    run_num    : sequential run counter across all experiments
    model      : OpenAI model ID string
    experiment : label string for which experiment this run belongs to
    messages   : input message list (defaults to global MESSAGES)
    inst_c     : instructor client (defaults to global inst_client)
    raw_c      : raw OpenAI client (defaults to global raw_client)
    trc        : tracer (defaults to global tracer)
    """
    if messages is None: messages = MESSAGES
    if inst_c   is None: inst_c   = inst_client
    if raw_c    is None: raw_c    = raw_client
    if trc      is None: trc      = tracer

    agent = TokenTrackedAgent(inst_c, raw_c, trc, model)

    error_msg      = ""
    render_success = False
    render_latency = 0.0
    chart_type     = ""
    cfg            = None

    try:
        ret = agent.start_main_span(messages)

        # Extract chart config from tool result message
        try:
            if isinstance(ret, dict) and "content" in ret:
                cfg = json.loads(ret["content"])
            elif hasattr(ret, "content"):
                c = ret.content
                cfg = json.loads(c) if isinstance(c, str) else None
        except Exception:
            cfg = None

        if cfg:
            ct = cfg.get("charttype", "")
            chart_type = ct.value if hasattr(ct, "value") else str(ct)

    except Exception as e:
        error_msg = str(e)

    # ── Render stage (optional) ────────────────────────────────────────
    if RENDER_AVAILABLE and cfg and not error_msg:
        try:
            plt.close("all")
            cfg_copy = copy.deepcopy(cfg)
            ct = cfg_copy.get("charttype", "bar")
            cfg_copy["charttype"] = ct.value if hasattr(ct, "value") else ct
            # Normalise data groups
            nd = []
            for g in (cfg_copy.get("data") or []):
                d  = g if isinstance(g, dict) else g.model_dump()
                xd = d.get("x_data", [])
                yd = [float(v) for v in (d.get("y_data") or [])]
                if not isinstance(xd, list): xd = list(range(len(yd)))
                n  = min(len(xd), len(yd))
                nd.append({"label": d.get("label", ""), "x_data": xd[:n], "y_data": yd[:n]})
            cfg_copy["data"] = nd
            cfg_copy.setdefault("annotations", [])

            t0 = time.perf_counter()
            generate_from_template(cfg_copy)
            render_latency = time.perf_counter() - t0
            render_success = True
            plt.close("all")
        except Exception as e:
            render_latency = 0.0
            render_success = False
            if not error_msg:
                error_msg = f"render_error: {e}"

    # ── Collect timings and tokens ─────────────────────────────────────
    timings  = dict(agent.timings)
    tok_all  = agent.get_all_stage_tokens()
    per_s    = tok_all["per_stage"]
    total_t  = tok_all["total"]

    def _t(stage, key="prompt"):
        return per_s.get(stage, {}).get(key, 0)

    return {
        "run_number":             run_num,
        "experiment":             experiment,
        "model":                  model,
        # ── Latency ──────────────────────────────────────────────────
        "total_latency_s":        round(timings.get("total", 0), 3),
        "router_latency_s":       round(timings.get("router", 0), 3),
        "analyze_latency_s":      round(timings.get("analyze_data", 0), 3),
        "extract_latency_s":      round(timings.get("extract_chart_config", 0), 3),
        "justify_latency_s":      round(timings.get("justify_chart_type", 0), 3),
        "render_latency_s":       round(render_latency, 3),
        # ── Router tokens ────────────────────────────────────────────
        "router_prompt_tokens":      _t("router",               "prompt"),
        "router_completion_tokens":  _t("router",               "completion"),
        "router_reasoning_tokens":   _t("router",               "reasoning"),
        # ── Analyze tokens ────────────────────────────────────────────
        "analyze_prompt_tokens":     _t("analyze_data",         "prompt"),
        "analyze_completion_tokens": _t("analyze_data",         "completion"),
        "analyze_reasoning_tokens":  _t("analyze_data",         "reasoning"),
        # ── Extract tokens ────────────────────────────────────────────
        "extract_prompt_tokens":     _t("extract_chart_config", "prompt"),
        "extract_completion_tokens": _t("extract_chart_config", "completion"),
        "extract_reasoning_tokens":  _t("extract_chart_config", "reasoning"),
        # ── Justify tokens ────────────────────────────────────────────
        "justify_prompt_tokens":     _t("justify_chart_type",   "prompt"),
        "justify_completion_tokens": _t("justify_chart_type",   "completion"),
        "justify_reasoning_tokens":  _t("justify_chart_type",   "reasoning"),
        # ── Totals ────────────────────────────────────────────────────
        "total_input_tokens":        total_t["prompt"],
        "total_output_tokens":       total_t["completion"],
        "total_reasoning_tokens":    total_t["reasoning"],
        "total_tokens":              total_t["total"],
        # ── Outcome ───────────────────────────────────────────────────
        "render_success":  render_success,
        "chart_type":      chart_type,
        "error_message":   error_msg,
    }

print("✓ run_one() helper defined")

✓ run_one() helper defined


---
## Experiment 1 — Single Run Bottleneck Breakdown
One run with full detail. Shows exactly where time goes and how many tokens each step uses.
Preserves `TimedAgent.print_bottleneck_table()` output format.

In [4]:
# ── Experiment 1: Single run ──────────────────────────────────────────
print(f"Model: {MODEL}  |  {'REASONING MODEL' if MODEL in REASONING_MODELS else 'STANDARD MODEL'}")
print("Running pipeline (1 run)...\n")

_run_counter = 1
row1 = run_one(_run_counter, MODEL, "exp1_single_run")
ALL_RESULTS.append(row1)

# ── Original-style bottleneck table (from TimedAgent) ─────────────────
# Re-create a TimedAgent just for the formatted table (read-only)
_agent_exp1 = TokenTrackedAgent(inst_client, raw_client, tracer, MODEL)
_agent_exp1.timings = OrderedDict({
    "router":               row1["router_latency_s"],
    "analyze_data":         row1["analyze_latency_s"],
    "extract_chart_config": row1["extract_latency_s"],
    "justify_chart_type":   row1["justify_latency_s"],
    "framework_overhead":   max(0, row1["total_latency_s"] - row1["router_latency_s"]
                                - row1["analyze_latency_s"] - row1["extract_latency_s"]
                                - row1["justify_latency_s"]),
    "total":                row1["total_latency_s"],
})
_agent_exp1.token_usage = OrderedDict({
    "router":               {"prompt_tokens": row1["router_prompt_tokens"]},
    "analyze_data":         {"prompt_tokens": row1["analyze_prompt_tokens"]},
    "extract_chart_config": {"prompt_tokens": row1["extract_prompt_tokens"]},
    "justify_chart_type":   {"prompt_tokens": row1["justify_prompt_tokens"]},
})
_agent_exp1.print_bottleneck_table()

# ── Token breakdown table ──────────────────────────────────────────────
STAGES_EXP = [
    ("router",               "router"),
    ("analyze_data",         "analyze"),
    ("extract_chart_config", "extract"),
    ("justify_chart_type",   "justify"),
]

print(f"\n{'Stage':<24} {'Prompt':>9} {'Completion':>12} {'Reasoning':>11} {'Total':>8}")
print("-" * 68)
for stage_key, csv_key in STAGES_EXP:
    p  = row1[f"{csv_key}_prompt_tokens"]
    c  = row1[f"{csv_key}_completion_tokens"]
    r  = row1[f"{csv_key}_reasoning_tokens"]
    print(f"{stage_key:<24} {p:>9} {c:>12} {r:>11} {p+c:>8}")
print("-" * 68)
print(f"{'TOTAL':<24} {row1['total_input_tokens']:>9} {row1['total_output_tokens']:>12} "
      f"{row1['total_reasoning_tokens']:>11} {row1['total_tokens']:>8}")

print(f"\nRender: {'✓ success' if row1['render_success'] else '✗ skipped/failed'}  "
      f"| chart_type: {row1['chart_type']}  | error: {row1['error_message'] or 'none'}")

Model: o4-mini  |  REASONING MODEL
Running pipeline (1 run)...

Starting main span with messages: [{'role': 'user', 'content': 'Wieviel Umsatz hatte Teckentrup in den Jahren 2021 bis 2024 im Segment JVA? Provide a detailed analysis of the data based on total earnings and provide a comprehensive visualization supporting your analysis.'}, {'role': 'system', 'content': 'The data is: \n        |   Kalender[Jahr] | Kalender[Monat]   |   [Umsatz SD/CO] |\n        |-----------------:|:------------------|-----------------:|\n        |             2021 | Apr               |         30264.05 |\n        |             2021 | Aug               |          9660.99 |\n        |             2021 | Dez               |         31104.06 |\n        |             2021 | Feb               |         11619.17 |\n        |             2021 | Jan               |         12308.10 |\n        |             2021 | Jul               |          4399.77 |\n        |             2021 | Jun               |         12705.

Exception while exporting Span.
Traceback (most recent call last):
  File "/Users/srujanakadambari/anaconda3/envs/thesis/lib/python3.11/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/srujanakadambari/anaconda3/envs/thesis/lib/python3.11/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/Users/srujanakadambari/anaconda3/envs/thesis/lib/python3.11/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
ConnectionRefusedError: [Errno 61] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/srujanakadambari/anaconda3/envs/thesis/lib/python3.11/site-packages/urllib3/connectionpool.py", line 793, in urlopen
    response = self._make_request(
               ^^^^^^^^^^^^^^^^^^^
  File "/Users/srujanakadambari/anaconda

generate_visualization
Main span completed with return value: {'role': 'tool', 'content': '{"titlename": "Umsatz JVA-Segment 2021\\u20132024 (Monatlicher Beitrag und Jahresvergleich)", "charttype": "line", "xlabel": "Jahr", "ylabel": "Umsatz (\\u20ac)", "y_ticks": [0, 50000, 100000, 150000, 200000, 250000, 300000], "x_ticks": [2021, 2022, 2023, 2024], "x_tick_label": ["2021", "2022", "2023", "2024"], "y_tick_label": ["0", "50k", "100k", "150k", "200k", "250k", "300k"], "x_lim": [2020.5, 2024.5], "y_lim": [0, 300000], "annotations": [{"text": "Spitzenmonat: Juni 2022", "target_type": "point", "data_id": 5.0, "data_value": 242067.31, "x_value": 2022.0, "y_value": 242067.31, "x_range": null, "y_range": null, "style": {"color": "red", "arrow": true}}, {"text": "Null-Umsatz: September 2024", "target_type": "point", "data_id": 8.0, "data_value": 0.0, "x_value": 2024.0, "y_value": 0.0, "x_range": null, "y_range": null, "style": {"color": "blue", "arrow": true}}, {"text": "Anm. Saisonmuster: H

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Experiment 2 — Repeated Runs (N=5) with Per-Stage Averages
One run can be noisy due to OpenAI server queue variance. Five runs give stable stage averages.

In [5]:
# ── Experiment 2: 5 repeated runs ────────────────────────────────────
N2 = 5
print(f"Model: {MODEL} | {N2} runs | No cache")
print("-" * 65)

exp2_rows = []
for i in range(N2):
    _run_counter += 1
    row = run_one(_run_counter, MODEL, "exp2_repeated")
    exp2_rows.append(row)
    ALL_RESULTS.append(row)
    print(f"Run {i+1}/{N2}: total={row['total_latency_s']:.2f}s  "
          f"router={row['router_latency_s']:.1f}  "
          f"analyze={row['analyze_latency_s']:.1f}  "
          f"extract={row['extract_latency_s']:.1f}  "
          f"justify={row['justify_latency_s']:.1f}  "
          f"reasoning_tok={row['total_reasoning_tokens']}")

avg_total = statistics.mean([r["total_latency_s"] for r in exp2_rows])

latency_keys = [
    ("router",               "router_latency_s",  "router_reasoning_tokens"),
    ("analyze_data",         "analyze_latency_s", "analyze_reasoning_tokens"),
    ("extract_chart_config", "extract_latency_s", "extract_reasoning_tokens"),
    ("justify_chart_type",   "justify_latency_s", "justify_reasoning_tokens"),
    ("total",                "total_latency_s",   "total_reasoning_tokens"),
]

print(f"\n{'Stage':<24} {'Avg (s)':>9} {'Std (s)':>9} {'% Total':>9} {'Avg Reason Tok':>16}")
print("-" * 70)
for label, lat_col, rtok_col in latency_keys:
    vals  = [r[lat_col] for r in exp2_rows]
    avg   = statistics.mean(vals)
    std   = statistics.stdev(vals) if len(vals) > 1 else 0.0
    pct   = avg / avg_total * 100 if avg_total > 0 else 0
    rtok  = statistics.mean([r[rtok_col] for r in exp2_rows])
    print(f"{label:<24} {avg:>9.2f} {std:>9.2f} {pct:>8.1f}%  {rtok:>14.0f}")

llm_keys = ["router_latency_s", "analyze_latency_s", "extract_latency_s", "justify_latency_s"]
llm_avg  = sum(statistics.mean([r[k] for r in exp2_rows]) for k in llm_keys)
print(f"\n→ LLM calls = {llm_avg/avg_total*100:.1f}% of total latency")
print(f"  Framework overhead = {(avg_total-llm_avg)/avg_total*100:.1f}% of total latency")

Model: o4-mini | 5 runs | No cache
-----------------------------------------------------------------
Starting main span with messages: [{'role': 'user', 'content': 'Wieviel Umsatz hatte Teckentrup in den Jahren 2021 bis 2024 im Segment JVA? Provide a detailed analysis of the data based on total earnings and provide a comprehensive visualization supporting your analysis.'}, {'role': 'system', 'content': 'The data is: \n        |   Kalender[Jahr] | Kalender[Monat]   |   [Umsatz SD/CO] |\n        |-----------------:|:------------------|-----------------:|\n        |             2021 | Apr               |         30264.05 |\n        |             2021 | Aug               |          9660.99 |\n        |             2021 | Dez               |         31104.06 |\n        |             2021 | Feb               |         11619.17 |\n        |             2021 | Jan               |         12308.10 |\n        |             2021 | Jul               |          4399.77 |\n        |             2021

Exception while exporting Span.
Traceback (most recent call last):
  File "/Users/srujanakadambari/anaconda3/envs/thesis/lib/python3.11/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/srujanakadambari/anaconda3/envs/thesis/lib/python3.11/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/Users/srujanakadambari/anaconda3/envs/thesis/lib/python3.11/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
ConnectionRefusedError: [Errno 61] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/srujanakadambari/anaconda3/envs/thesis/lib/python3.11/site-packages/urllib3/connectionpool.py", line 793, in urlopen
    response = self._make_request(
               ^^^^^^^^^^^^^^^^^^^
  File "/Users/srujanakadambari/anaconda

generate_visualization
Main span completed with return value: {'role': 'tool', 'content': '{"titlename": "Monatlicher Umsatz SD/CO nach Jahr (2021-2024)", "charttype": "line", "xlabel": "Monat", "ylabel": "Umsatz SD/CO (\\u20ac)", "y_ticks": [0, 50000, 100000, 150000, 200000, 250000], "x_ticks": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], "x_tick_label": ["Jan", "Feb", "Mrz", "Apr", "Mai", "Jun", "Jul", "Aug", "Sep", "Okt", "Nov", "Dez"], "y_tick_label": ["0", "50k", "100k", "150k", "200k", "250k"], "x_lim": [1, 12], "y_lim": [0, 250000], "annotations": [{"text": "H\\u00f6chster Monat 2022 (Juni)", "target_type": "point", "data_id": 1.0, "data_value": 242067.31, "x_value": 6.0, "y_value": 242067.31, "x_range": null, "y_range": null, "style": {"arrowstyle": "->", "color": "red"}}, {"text": "Spitzen-Umsatz 2021 (Nov)", "target_type": "point", "data_id": 0.0, "data_value": 43170.69, "x_value": 11.0, "y_value": 43170.69, "x_range": null, "y_range": null, "style": {"arrowstyle": "->", "color":

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


generate_visualization
Main span completed with return value: {'role': 'tool', 'content': '{"titlename": "J\\u00e4hrliche Ums\\u00e4tze SD/CO 2021\\u20132024", "charttype": "line", "xlabel": "Jahr", "ylabel": "Umsatz (\\u20ac)", "y_ticks": [0, 250000, 500000, 750000, 1000000, 1250000], "x_ticks": [0, 1, 2, 3], "x_tick_label": ["2021", "2022", "2023", "2024"], "y_tick_label": ["0", "250 T\\u20ac", "500 T\\u20ac", "750 T\\u20ac", "1,0 M\\u20ac", "1,25 M\\u20ac"], "x_lim": [-0.5, 3.5], "y_lim": [0, 1250000], "annotations": [{"text": "+448 % (2021\\u21922022)", "target_type": "point", "data_id": 1.0, "data_value": 1124838.0, "x_value": 1.0, "y_value": 1124838.0, "x_range": null, "y_range": null, "style": {"va": "bottom", "ha": "center", "color": "darkgreen"}}, {"text": "\\u201342 % (2022\\u21922023)", "target_type": "point", "data_id": 2.0, "data_value": 656641.0, "x_value": 2.0, "y_value": 656641.0, "x_range": null, "y_range": null, "style": {"va": "bottom", "ha": "center", "color": "dark

...

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


generate_visualization
Main span completed with return value: {'role': 'tool', 'content': '{"titlename": "Monatlicher Umsatz je Jahr (Segment JVA)", "charttype": "line", "xlabel": "Monat", "ylabel": "Umsatz SD/CO (\\u20ac)", "y_ticks": [0, 50000, 100000, 150000, 200000, 250000], "x_ticks": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], "x_tick_label": ["Jan", "Feb", "Mrz", "Apr", "Mai", "Jun", "Jul", "Aug", "Sep", "Okt", "Nov", "Dez"], "y_tick_label": ["0", "50k", "100k", "150k", "200k", "250k"], "x_lim": [1, 12], "y_lim": [0, 260000], "annotations": [{"text": "H\\u00f6chster Umsatz im Juni 2022 (242k EUR)", "target_type": "point", "data_id": 1.0, "data_value": 242067.31, "x_value": 6.0, "y_value": 242067.31, "x_range": null, "y_range": null, "style": {"color": "red", "fontsize": 10, "arrowstyle": "->"}}, {"text": "Spitze im Nov 2022 (191.8k EUR)", "target_type": "point", "data_id": 1.0, "data_value": 191821.13, "x_value": 11.0, "y_value": 191821.13, "x_range": null, "y_range": null, "style"

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/srujanakadambari/anaconda3/envs/thesis/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `list[float]` - serialized value may not be as expected [field_name='x_data', input_value='Array for data on x-Axis', input_type=str])
  PydanticSerializationUnexpectedValue(Expected `list[float]` - serialized value may not be as expected [field_name='x_data', input_value='Array for data on x-Axis', input_type=str])
  PydanticSerializationUnexpectedValue(Expected `list[float]` - serialized value may not be as expected [field_name='x_data', input_value='Array for data on x-Axis', input_type=str])
  PydanticSerializationUnexpectedValue(Expected `list[float]` - serialized value may not be as expe

generate_visualization
Main span completed with return value: {'role': 'tool', 'content': '{"titlename": "Jahresumsatzvergleich 2021\\u20132024", "charttype": "line", "xlabel": "Jahr", "ylabel": "Umsatz in \\u20ac", "y_ticks": [0, 250000, 500000, 750000, 1000000, 1250000], "x_ticks": [0, 1, 2, 3], "x_tick_label": ["2021", "2022", "2023", "2024"], "y_tick_label": ["0", "250 T\\u20ac", "500 T\\u20ac", "750 T\\u20ac", "1 M\\u20ac", "1,25 M\\u20ac"], "x_lim": [-0.5, 3.5], "y_lim": [0, 1300000], "annotations": [{"text": "Starker Anstieg 2022 (+449 % gg\\u00fc. 2021)", "target_type": "point", "data_id": 1.0, "data_value": 1124838.11, "x_value": null, "y_value": null, "x_range": null, "y_range": null, "style": {"arrow": {"color": "black", "dx": 0, "dy": -20}, "fontsize": 10}}, {"text": "R\\u00fcckgang 2023 (-42 % gg\\u00fc. 2022)", "target_type": "point", "data_id": 2.0, "data_value": 656640.59, "x_value": null, "y_value": null, "x_range": null, "y_range": null, "style": {"arrow": {"color": "

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Experiment 3 — Model Comparison: o4-mini vs gpt-4o-mini
Same pipeline, same data, different model.
`o4-mini` = reasoning model (generates internal thinking tokens — these are the major source of variance).
`gpt-4o-mini` = standard model (no reasoning tokens, more predictable latency).

In [6]:
# ── Experiment 3: model comparison ────────────────────────────────────
COMPARE_MODELS = ["o4-mini", "gpt-4o-mini"]
N3 = 3
exp3 = {}

for model_name in COMPARE_MODELS:
    label = "REASONING" if model_name in REASONING_MODELS else "STANDARD"
    print(f"\n--- {model_name} [{label}] ---")
    runs = []
    for i in range(N3):
        _run_counter += 1
        row = run_one(_run_counter, model_name, "exp3_model_comparison")
        runs.append(row)
        ALL_RESULTS.append(row)
        print(f"  Run {i+1}/{N3}: {row['total_latency_s']:.2f}s  "
              f"router={row['router_latency_s']:.1f}  "
              f"analyze={row['analyze_latency_s']:.1f}  "
              f"extract={row['extract_latency_s']:.1f}  "
              f"justify={row['justify_latency_s']:.1f}  "
              f"reasoning_tok={row['total_reasoning_tokens']}")
    exp3[model_name] = runs
    totals = [r["total_latency_s"] for r in runs]
    rtoks  = [r["total_reasoning_tokens"] for r in runs]
    print(f"  → avg={statistics.mean(totals):.2f}s  "
          f"std={statistics.stdev(totals) if N3 > 1 else 0:.2f}s  "
          f"avg_reasoning_tok={statistics.mean(rtoks):.0f}")

# ── Per-stage comparison table ─────────────────────────────────────────
comp_stages = [
    ("router",               "router_latency_s"),
    ("analyze_data",         "analyze_latency_s"),
    ("extract_chart_config", "extract_latency_s"),
    ("justify_chart_type",   "justify_latency_s"),
    ("total",                "total_latency_s"),
]

print(f"\n{'Stage':<24}", end="")
for m in COMPARE_MODELS:
    print(f"  {m:>15}", end="")
print(f"  {'Δ (o4-gpt4mini)':>17}")
print("-" * 75)
for stage_lbl, lat_col in comp_stages:
    avgs = [statistics.mean([r[lat_col] for r in exp3[m]]) for m in COMPARE_MODELS]
    diff = avgs[0] - avgs[1]
    print(f"{stage_lbl:<24}", end="")
    for a in avgs:
        print(f"  {a:>14.2f}s", end="")
    print(f"  {diff:>+16.2f}s")

print(f"\n→ Δ > 0: o4-mini is slower due to reasoning tokens")
print(f"→ Δ at router stage = reasoning overhead in tool-call decision")
print(f"→ Δ at analyze/extract/justify = reasoning overhead in structured output stages")


--- o4-mini [REASONING] ---
Starting main span with messages: [{'role': 'user', 'content': 'Wieviel Umsatz hatte Teckentrup in den Jahren 2021 bis 2024 im Segment JVA? Provide a detailed analysis of the data based on total earnings and provide a comprehensive visualization supporting your analysis.'}, {'role': 'system', 'content': 'The data is: \n        |   Kalender[Jahr] | Kalender[Monat]   |   [Umsatz SD/CO] |\n        |-----------------:|:------------------|-----------------:|\n        |             2021 | Apr               |         30264.05 |\n        |             2021 | Aug               |          9660.99 |\n        |             2021 | Dez               |         31104.06 |\n        |             2021 | Feb               |         11619.17 |\n        |             2021 | Jan               |         12308.10 |\n        |             2021 | Jul               |          4399.77 |\n        |             2021 | Jun               |         12705.90 |\n        |             2021 | 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


generate_visualization
Main span completed with return value: {'role': 'tool', 'content': '{"titlename": "Monatlicher Umsatz SD/CO (2021\\u20132024)", "charttype": "line", "xlabel": "Monat (Index)", "ylabel": "Umsatz SD/CO (\\u20ac)", "y_ticks": [0, 50000, 100000, 150000, 200000, 250000], "x_ticks": [0, 11, 23, 35, 47], "x_tick_label": ["Jan 2021", "Dez 2021", "Dez 2022", "Dez 2023", "Dez 2024"], "y_tick_label": ["0", "50k", "100k", "150k", "200k", "250k"], "x_lim": [0, 47], "y_lim": [0, 260000], "annotations": [{"text": "Top 1: 242067.31 \\u20ac (Jun 2022)", "target_type": "point", "data_id": 17.0, "data_value": 242067.31, "x_value": 17.0, "y_value": 242067.31, "x_range": null, "y_range": null, "style": {"color": "red", "arrowprops": {"arrowstyle": "->", "color": "red"}}}, {"text": "Top 2: 191821.13 \\u20ac (Nov 2022)", "target_type": "point", "data_id": 22.0, "data_value": 191821.13, "x_value": 22.0, "y_value": 191821.13, "x_range": null, "y_range": null, "style": {"color": "orange",

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


generate_visualization
Main span completed with return value: {'role': 'tool', 'content': '{"titlename": "Monatliche Umsatzentwicklung im Segment JVA (2021\\u20132024)", "charttype": "line", "xlabel": "Monat", "ylabel": "Umsatz SD/CO (Euro)", "y_ticks": [0, 50000, 100000, 150000, 200000, 250000], "x_ticks": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], "x_tick_label": ["Jan", "Feb", "Mrz", "Apr", "Mai", "Jun", "Jul", "Aug", "Sep", "Okt", "Nov", "Dez"], "y_tick_label": ["0", "50k", "100k", "150k", "200k", "250k"], "x_lim": [1, 12], "y_lim": [0, 250000], "annotations": [{"text": "Spitzenmonat Juni 2022", "target_type": "point", "data_id": 1.0, "data_value": null, "x_value": 6.0, "y_value": 242067.31, "x_range": null, "y_range": null, "style": null}, {"text": "H\\u00f6chstwert Apr 2023", "target_type": "point", "data_id": 2.0, "data_value": null, "x_value": 4.0, "y_value": 136944.76, "x_range": null, "y_range": null, "style": null}, {"text": "Starker Sommerpeak Aug 2024", "target_type": "point

...

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


generate_visualization
generate_visualization
Main span completed with return value: {'role': 'tool', 'content': '{"titlename": "Monthly Revenue for Teckentrup (2021-2024)", "charttype": "line", "xlabel": "Month", "ylabel": "Revenue (SD/CO)", "y_ticks": [0, 50000, 100000, 150000, 200000, 250000], "x_ticks": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11], "x_tick_label": ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dez"], "y_tick_label": ["0", "50000", "100000", "150000", "200000", "250000"], "x_lim": [0, 11], "y_lim": [0, 250000], "annotations": [{"text": "Significant Revenue Growth: Jun 2022 revenue reached 242,067.31", "target_type": "point", "data_id": 18.0, "data_value": 242067.31, "x_value": null, "y_value": null, "x_range": null, "y_range": null, "style": null}, {"text": "Peak Revenue: May 2022 revenue is high indicating potential seasonality", "target_type": "point", "data_id": 17.0, "data_value": 79558.31, "x_value": null, "y_value": null, "x_range": n

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


generate_visualization
Main span completed with return value: {'role': 'tool', 'content': '{"titlename": "Revenue from Teckentrup in the JVA segment (2021-2024)", "charttype": "line", "xlabel": "Year", "ylabel": "Revenue (SD/CO)", "y_ticks": [0, 250000, 500000, 750000, 1000000, 1250000], "x_ticks": [2021, 2022, 2023, 2024], "x_tick_label": ["2021", "2022", "2023", "2024"], "y_tick_label": ["0", "250,000", "500,000", "750,000", "1,000,000", "1,250,000"], "x_lim": [2021, 2024], "y_lim": [0, 1300000], "annotations": [{"text": "Significant revenue growth (~462%) from 2021 to 2022", "target_type": "point", "data_id": 0.0, "data_value": 223282.63, "x_value": 2021.0, "y_value": 223282.63, "x_range": null, "y_range": null, "style": {"color": "blue", "fontsize": 10}}, {"text": "Revenue drops ~61% in 2023", "target_type": "point", "data_id": 1.0, "data_value": 1255583.75, "x_value": 2022.0, "y_value": 1255583.75, "x_range": null, "y_range": null, "style": {"color": "red", "fontsize": 10}}, {"tex

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Experiment 4 — Prompt Size Scaling
Does feeding more data rows into the prompt increase latency?
Tests 3 data sizes: ~1 year (~12 rows), ~2 years (~24 rows), full 4 years (~48 rows).

In [7]:
# ── Experiment 4: prompt size scaling ─────────────────────────────────
N4 = 2   # runs per size (6 API calls total — keep short)

lines      = MD_TABLE.strip().split("\n")
header     = "\n".join(lines[:2])
data_lines = lines[2:]

SIZES = {
    "small  (~1yr,  12 rows)": data_lines[:12],
    "medium (~2yr,  24 rows)": data_lines[:24],
    "large  (full,  48 rows)": data_lines[:48],
}

exp4 = {}
print(f"Model: {MODEL} | {N4} runs per size")
print("-" * 65)

for size_label, rows in SIZES.items():
    table_variant = header + "\n" + "\n".join(rows)
    msgs_variant  = [
        {"role": "user",   "content": QUESTION},
        {"role": "system", "content": f"The data is: {table_variant}"},
    ]
    approx_tokens = (len(QUESTION) + len(table_variant)) // 4
    size_runs = []

    for i in range(N4):
        _run_counter += 1
        row = run_one(_run_counter, MODEL, f"exp4_prompt_size",
                      messages=msgs_variant)
        size_runs.append(row)
        ALL_RESULTS.append(row)

    avg_lat = statistics.mean([r["total_latency_s"] for r in size_runs])
    exp4[size_label] = {"tokens": approx_tokens, "avg_s": avg_lat, "runs": size_runs}
    print(f"{size_label}: ~{approx_tokens:>5} tokens → avg {avg_lat:.2f}s")

# ── Scaling table ──────────────────────────────────────────────────────
labels = list(exp4.keys())
base   = exp4[labels[0]]["avg_s"]
print(f"\n{'Size':<30} {'~Tokens':>8} {'Avg (s)':>9} {'Δ from small':>14}")
print("-" * 64)
for lbl, d in exp4.items():
    delta = d["avg_s"] - base
    print(f"{lbl:<30} {d['tokens']:>8} {d['avg_s']:>9.2f} {delta:>+13.2f}s")

print(f"\n→ Positive Δ per extra row = additional prefill cost at the LLM side")
print(f"→ Steep slope suggests prompt tokens drive latency for {MODEL}")

Model: o4-mini | 2 runs per size
-----------------------------------------------------------------
Starting main span with messages: [{'role': 'user', 'content': 'Wieviel Umsatz hatte Teckentrup in den Jahren 2021 bis 2024 im Segment JVA? Provide a detailed analysis of the data based on total earnings and provide a comprehensive visualization supporting your analysis.'}, {'role': 'system', 'content': 'The data is: |   Kalender[Jahr] | Kalender[Monat]   |   [Umsatz SD/CO] |\n        |-----------------:|:------------------|-----------------:|\n        |             2021 | Apr               |         30264.05 |\n        |             2021 | Aug               |          9660.99 |\n        |             2021 | Dez               |         31104.06 |\n        |             2021 | Feb               |         11619.17 |\n        |             2021 | Jan               |         12308.10 |\n        |             2021 | Jul               |          4399.77 |\n        |             2021 | Jun      

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


generate_visualization
Main span completed with return value: {'role': 'tool', 'content': '{"titlename": "Monatliche Umsatzentwicklung SD/CO 2021", "charttype": "line", "xlabel": "Monat", "ylabel": "Umsatz (Euro)", "y_ticks": [0, 10000, 20000, 30000, 40000, 50000], "x_ticks": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], "x_tick_label": ["Jan", "Feb", "Mrz", "Apr", "Mai", "Jun", "Jul", "Aug", "Sep", "Okt", "Nov", "Dez"], "y_tick_label": ["0", "10.000", "20.000", "30.000", "40.000", "50.000"], "x_lim": [1, 12], "y_lim": [0, 50000], "annotations": [{"text": "H\\u00f6chster Umsatz", "target_type": "point", "data_id": 10.0, "data_value": 43170.69, "x_value": null, "y_value": null, "x_range": null, "y_range": null, "style": {"color": "red", "arrow": true}}, {"text": "Zweiter Top-Monat", "target_type": "point", "data_id": 11.0, "data_value": 31104.06, "x_value": null, "y_value": null, "x_range": null, "y_range": null, "style": {"color": "darkred", "arrow": true}}, {"text": "Spitzenwert Fr\\u00fch

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


generate_visualization
Main span completed with return value: {'role': 'tool', 'content': '{"titlename": "Umsatz SD/CO nach Monat und Jahr", "charttype": "line", "xlabel": "Monat", "ylabel": "Umsatz (\\u20ac)", "y_ticks": [0, 50000, 100000, 150000, 200000, 250000], "x_ticks": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], "x_tick_label": ["Jan", "Feb", "Mrz", "Apr", "Mai", "Jun", "Jul", "Aug", "Sep", "Okt", "Nov", "Dez"], "y_tick_label": ["0", "50k", "100k", "150k", "200k", "250k"], "x_lim": [1, 12], "y_lim": [0, 260000], "annotations": [{"text": "H\\u00f6chster Einzelmonat 2022: \\u20ac242.067", "target_type": "point", "data_id": 5.0, "data_value": 242067.31, "x_value": null, "y_value": null, "x_range": null, "y_range": null, "style": {"color": "red", "arrow": true}}, {"text": "2021 Gesamt: \\u20ac204.872,32", "target_type": "chart", "data_id": null, "data_value": null, "x_value": null, "y_value": null, "x_range": null, "y_range": null, "style": null}, {"text": "2022 Gesamt: \\u20ac1.124.83

...

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


generate_visualization
Main span completed with return value: {'role': 'tool', 'content': '{"titlename": "Monatlicher Umsatz SD/CO nach Jahr (2021\\u20132024)", "charttype": "line", "xlabel": "Monat", "ylabel": "Umsatz (\\u20ac)", "y_ticks": [0, 50000, 100000, 150000, 200000, 250000, 300000], "x_ticks": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], "x_tick_label": ["Jan", "Feb", "Mrz", "Apr", "Mai", "Jun", "Jul", "Aug", "Sep", "Okt", "Nov", "Dez"], "y_tick_label": ["0", "50k", "100k", "150k", "200k", "250k", "300k"], "x_lim": [1, 12], "y_lim": [0, 260000], "annotations": [{"text": "H\\u00f6chster Monat: Juni 2022 (242 067 \\u20ac)", "target_type": "point", "data_id": null, "data_value": null, "x_value": 6.0, "y_value": 242067.31, "x_range": null, "y_range": null, "style": {"arrowstyle": "->", "color": "red", "fontsize": 10}}, {"text": "Null-Umsatz: September 2024", "target_type": "point", "data_id": null, "data_value": null, "x_value": 9.0, "y_value": 0.0, "x_range": null, "y_range": null, 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


generate_visualization
Main span completed with return value: {'role': 'tool', 'content': '{"titlename": "\\u00dcbersicht der Ums\\u00e4tze Teckentrup JVA 2021\\u20132024", "charttype": "line", "xlabel": "Jahr", "ylabel": "Umsatz (EUR)", "y_ticks": [0, 250000, 500000, 750000, 1000000, 1250000], "x_ticks": [2021, 2022, 2023, 2024], "x_tick_label": ["2021", "2022", "2023", "2024"], "y_tick_label": ["0", "0,25M", "0,5M", "0,75M", "1,0M", "1,25M"], "x_lim": [2020.8, 2024.2], "y_lim": [0, 1300000], "annotations": [{"text": "Starker Anstieg: +449% gg\\u00fc. 2021", "target_type": "point", "data_id": 1.0, "data_value": 1124838.11, "x_value": 2022.0, "y_value": 1124838.11, "x_range": null, "y_range": null, "style": {"color": "green", "arrow": true}}, {"text": "R\\u00fcckgang um 41,6% gg\\u00fc. 2022", "target_type": "point", "data_id": 2.0, "data_value": 656640.59, "x_value": 2023.0, "y_value": 656640.59, "x_range": null, "y_range": null, "style": {"color": "red", "arrow": true}}, {"text": "We

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Experiment 5 — API Variance + Reasoning Token Correlation

**This directly answers the advisor question about values falling between cold/warm start.**

10 identical runs. We measure:
- Total latency and per-stage latency per run
- Reasoning tokens generated per run
- Pearson correlation between reasoning tokens and total latency

If correlation is high (|r| > 0.7) → the variance is caused by **o4-mini generating different amounts of reasoning tokens per call**, not by cold/warm network effects.

In [8]:
# ── Experiment 5: variance + reasoning-token correlation ──────────────
N5 = 10
print(f"Model: {MODEL} (REASONING MODEL) | {N5} identical runs | No cache")
print()
print(f"{'Run':>4} {'Total':>8} {'Router':>8} {'Analyze':>9} "
      f"{'Extract':>9} {'Justify':>9} {'Reason Tok':>12} {'Compl Tok':>10}")
print("-" * 82)

exp5_rows = []
for i in range(N5):
    _run_counter += 1
    row = run_one(_run_counter, MODEL, "exp5_variance")
    exp5_rows.append(row)
    ALL_RESULTS.append(row)
    print(f"{i+1:>4} {row['total_latency_s']:>8.2f} {row['router_latency_s']:>8.2f} "
          f"{row['analyze_latency_s']:>9.2f} {row['extract_latency_s']:>9.2f} "
          f"{row['justify_latency_s']:>9.2f} "
          f"{row['total_reasoning_tokens']:>12} {row['total_output_tokens']:>10}")

totals = [r["total_latency_s"]        for r in exp5_rows]
rtoks  = [r["total_reasoning_tokens"]  for r in exp5_rows]
ctoks  = [r["total_output_tokens"]     for r in exp5_rows]

mean_t = statistics.mean(totals)
std_t  = statistics.stdev(totals)
cv     = std_t / mean_t * 100
mde    = std_t * 2

print(f"\n--- Variance Summary ---")
print(f"Mean:    {mean_t:.2f}s")
print(f"Stdev:   {std_t:.2f}s")
print(f"CV:      {cv:.1f}%")
print(f"Range:   {min(totals):.2f}s  —  {max(totals):.2f}s")
print(f"MDE:     {mde:.2f}s  (minimum detectable effect = 2 × stdev)")

print(f"\n--- Reasoning Token Distribution ---")
if rtoks:
    print(f"Min: {min(rtoks)}   Max: {max(rtoks)}   "
          f"Mean: {statistics.mean(rtoks):.0f}   "
          f"Std: {statistics.stdev(rtoks) if len(rtoks) > 1 else 0:.0f}")

# ── Pearson correlation: reasoning tokens vs latency ──────────────────
if len(set(rtoks)) > 1:
    n      = len(totals)
    mean_r = statistics.mean(rtoks)
    mean_l = statistics.mean(totals)
    cov    = sum((r - mean_r) * (l - mean_l) for r, l in zip(rtoks, totals)) / n
    std_r  = statistics.pstdev(rtoks)
    std_l  = statistics.pstdev(totals)
    corr   = cov / (std_r * std_l) if std_r > 0 and std_l > 0 else 0.0
    print(f"\nPearson r (reasoning_tokens vs latency) = {corr:.3f}")
    if abs(corr) > 0.7:
        print(f"→ STRONG correlation: reasoning token count is driving the latency spread.")
        print(f"  The {cv:.1f}% CV is NOT cold/warm start — it is reasoning token variance.")
    elif abs(corr) > 0.4:
        print(f"→ MODERATE correlation: reasoning tokens partially explain the spread.")
    else:
        print(f"→ WEAK correlation: network / OpenAI server queue variance dominates.")
else:
    print("\nNote: reasoning tokens were constant across all runs.")
    print("      The variance comes from network / OpenAI server queue noise.")
    corr = 0.0

print(f"""
--- Advisor explanation ---
Model used: {MODEL} (reasoning model — uses internal thinking tokens)

Values fall 'in between cold/warm start' because two variance sources overlap:
  1. Network warm-up: ~5-15 s one-time cost on the first call (TCP/TLS handshake)
  2. Reasoning token variance: {MODEL} generates variable reasoning tokens
     even for identical prompts → latency varies every single run

Source 1 is one-time per session. Source 2 is irreducible (not a cold/warm effect).
The Pearson correlation r={corr:.3f} indicates which source dominates.
""")

Model: o4-mini (REASONING MODEL) | 10 identical runs | No cache

 Run    Total   Router   Analyze   Extract   Justify   Reason Tok  Compl Tok
----------------------------------------------------------------------------------
Starting main span with messages: [{'role': 'user', 'content': 'Wieviel Umsatz hatte Teckentrup in den Jahren 2021 bis 2024 im Segment JVA? Provide a detailed analysis of the data based on total earnings and provide a comprehensive visualization supporting your analysis.'}, {'role': 'system', 'content': 'The data is: \n        |   Kalender[Jahr] | Kalender[Monat]   |   [Umsatz SD/CO] |\n        |-----------------:|:------------------|-----------------:|\n        |             2021 | Apr               |         30264.05 |\n        |             2021 | Aug               |          9660.99 |\n        |             2021 | Dez               |         31104.06 |\n        |             2021 | Feb               |         11619.17 |\n        |             2021 | Jan        

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


generate_visualization
Main span completed with return value: {'role': 'tool', 'content': '{"titlename": "Monatlicher Umsatz SD/CO (Januar 2021 \\u2013 Dezember 2024)", "charttype": "line", "xlabel": "Monat / Jahr", "ylabel": "Umsatz (EUR)", "y_ticks": [0, 50000, 100000, 150000, 200000, 250000], "x_ticks": [0, 12, 24, 36, 47], "x_tick_label": ["Jan 2021", "Jan 2022", "Jan 2023", "Jan 2024", "Dez 2024"], "y_tick_label": ["0", "50 000", "100 000", "150 000", "200 000", "250 000"], "x_lim": [0, 47], "y_lim": [0, 260000], "annotations": [{"text": "H\\u00f6chster Umsatz: 242 067,31 \\u20ac", "target_type": "point", "data_id": 17.0, "data_value": 242067.31, "x_value": 17.0, "y_value": 242067.31, "x_range": null, "y_range": null, "style": {"color": "red", "arrowprops": {"arrowstyle": "->"}}}, {"text": "Null-Umsatz: 0,00 \\u20ac", "target_type": "point", "data_id": 44.0, "data_value": 0.0, "x_value": 44.0, "y_value": 0.0, "x_range": null, "y_range": null, "style": {"color": "blue", "arrowprops

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


generate_visualization
Main span completed with return value: {'role': 'tool', 'content': '{"titlename": "Monatlicher Umsatz SD/CO nach Jahr (2021\\u20132024)", "charttype": "line", "xlabel": "Monat", "ylabel": "Umsatz SD/CO (\\u20ac)", "y_ticks": [0, 50000, 100000, 150000, 200000, 250000], "x_ticks": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], "x_tick_label": ["Jan", "Feb", "Mrz", "Apr", "Mai", "Jun", "Jul", "Aug", "Sep", "Okt", "Nov", "Dez"], "y_tick_label": ["0", "50k", "100k", "150k", "200k", "250k"], "x_lim": [1, 12], "y_lim": [0, 270000], "annotations": [{"text": "H\\u00f6chster Monat 2022", "target_type": "point", "data_id": 1.0, "data_value": 242067.31, "x_value": 6.0, "y_value": 242067.31, "x_range": null, "y_range": null, "style": {"color": "red", "arrow": true}}, {"text": "Ausrei\\u00dfer Dez 2023", "target_type": "point", "data_id": 2.0, "data_value": 308.46, "x_value": 12.0, "y_value": 308.46, "x_range": null, "y_range": null, "style": {"color": "orange", "arrow": true}}, {"t

...

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


generate_visualization
Main span completed with return value: {'role': 'tool', 'content': '{"titlename": "Monatlicher Umsatz SD/CO nach Jahren (2021\\u20132024)", "charttype": "line", "xlabel": "Monat", "ylabel": "Umsatz (\\u20ac)", "y_ticks": [0, 50000, 100000, 150000, 200000, 250000], "x_ticks": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], "x_tick_label": ["Jan", "Feb", "Mrz", "Apr", "Mai", "Jun", "Jul", "Aug", "Sep", "Okt", "Nov", "Dez"], "y_tick_label": [0, 50000, 100000, 150000, 200000, 250000], "x_lim": [1, 12], "y_lim": [0, 260000], "annotations": [{"text": "2022: H\\u00f6chster Umsatz (Juni: 242.067 \\u20ac)", "target_type": "point", "data_id": 1.0, "data_value": 242067.31, "x_value": 6.0, "y_value": 242067.31, "x_range": null, "y_range": null, "style": null}, {"text": "2024: Kein Umsatz (September)", "target_type": "point", "data_id": 3.0, "data_value": 0.0, "x_value": 9.0, "y_value": 0.0, "x_range": null, "y_range": null, "style": null}, {"text": "2023: Tiefstwert im Dezember (0,

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


generate_visualization
Main span completed with return value: {'role': 'tool', 'content': '{"titlename": "Umsatz SD/CO: Monatsverlauf 2021-2024", "charttype": "line", "xlabel": "Monat", "ylabel": "Umsatz SD/CO (\\u20ac)", "y_ticks": [0, 50000, 100000, 150000, 200000, 250000], "x_ticks": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], "x_tick_label": ["Jan", "Feb", "Mrz", "Apr", "Mai", "Jun", "Jul", "Aug", "Sep", "Okt", "Nov", "Dez"], "y_tick_label": ["0", "50 000", "100 000", "150 000", "200 000", "250 000"], "x_lim": [1, 12], "y_lim": [0, 250000], "annotations": [{"text": "Ausrei\\u00dfer: Juni 2022 (242 067 \\u20ac)", "target_type": "point", "data_id": 1.0, "data_value": 242067.31, "x_value": null, "y_value": null, "x_range": null, "y_range": null, "style": {"color": "red", "arrow": true, "fontsize": 10}}], "data": [{"x_data": [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0], "y_data": [12308.1, 11619.17, 6366.93, 30264.05, 4013.68, 12705.9, 4399.77, 9660.99, 8191.73, 31067.2

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## CSV Export — All Experiment Results

In [9]:
# ═══════════════════════════════════════════════════════════════════
# CSV Export — one row per run, all experiments combined
# ═══════════════════════════════════════════════════════════════════
import csv
from pathlib import Path

CSV_PATH = HERE / "bottleneck_thesis_results.csv"

if ALL_RESULTS:
    fieldnames = list(ALL_RESULTS[0].keys())
    with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(ALL_RESULTS)
    print(f"✅ CSV saved → {CSV_PATH}")
    print(f"   {len(ALL_RESULTS)} rows  |  {len(fieldnames)} columns")
    print(f"\nColumns:")
    for i, col in enumerate(fieldnames):
        print(f"  {i+1:>2}. {col}")
else:
    print("⚠  ALL_RESULTS is empty — run the experiment cells first")

✅ CSV saved → /Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/FINAL FOLDER/bottleneck_thesis_results.csv
   28 rows  |  28 columns

Columns:
   1. run_number
   2. experiment
   3. model
   4. total_latency_s
   5. router_latency_s
   6. analyze_latency_s
   7. extract_latency_s
   8. justify_latency_s
   9. render_latency_s
  10. router_prompt_tokens
  11. router_completion_tokens
  12. router_reasoning_tokens
  13. analyze_prompt_tokens
  14. analyze_completion_tokens
  15. analyze_reasoning_tokens
  16. extract_prompt_tokens
  17. extract_completion_tokens
  18. extract_reasoning_tokens
  19. justify_prompt_tokens
  20. justify_completion_tokens
  21. justify_reasoning_tokens
  22. total_input_tokens
  23. total_output_tokens
  24. total_reasoning_tokens
  25. total_tokens
  26. render_success
  27. chart_type
  28. error_message


---
## Summary Statistics
Mean, Median, Std, Min, Max, P90, P95, CV — computed over all runs of the primary model.

In [10]:
# ═══════════════════════════════════════════════════════════════════
# Summary Statistics (primary model, all experiments)
# ═══════════════════════════════════════════════════════════════════
import math

def _pct(data: list, p: float) -> float:
    """Percentile via linear interpolation."""
    if not data: return 0.0
    sd = sorted(data)
    k  = (len(sd) - 1) * p / 100
    lo, hi = int(k), math.ceil(k)
    return sd[lo] + (sd[hi] - sd[lo]) * (k - lo) if lo != hi else sd[lo]

def _stats(values: list) -> dict:
    if not values:
        return {"mean": 0, "median": 0, "std": 0, "min": 0,
                "max": 0, "P90": 0, "P95": 0, "CV%": 0}
    m = statistics.mean(values)
    return {
        "mean":   round(m, 3),
        "median": round(statistics.median(values), 3),
        "std":    round(statistics.stdev(values) if len(values) > 1 else 0, 3),
        "min":    round(min(values), 3),
        "max":    round(max(values), 3),
        "P90":    round(_pct(values, 90), 3),
        "P95":    round(_pct(values, 95), 3),
        "CV%":    round((statistics.stdev(values) / m * 100) if m > 0 and len(values) > 1 else 0, 1),
    }

primary_rows = [r for r in ALL_RESULTS if r["model"] == MODEL]

if not primary_rows:
    print("No results yet — run experiment cells first.")
else:
    stat_cols = [
        ("total_latency_s",   "Total latency (s)"),
        ("router_latency_s",  "  Router (s)"),
        ("analyze_latency_s", "  Analyze (s)"),
        ("extract_latency_s", "  Extract (s)"),
        ("justify_latency_s", "  Justify (s)"),
        ("render_latency_s",  "  Render (s)"),
        ("total_tokens",      "Total tokens"),
        ("total_reasoning_tokens", "Reasoning tokens"),
        ("total_input_tokens",     "Input tokens"),
        ("total_output_tokens",    "Output tokens"),
    ]

    header = f"{'Metric':<26} {'Mean':>10} {'Median':>8} {'Std':>8} {'Min':>8} {'Max':>8} {'P90':>8} {'P95':>8} {'CV%':>6}"
    print(f"Summary Statistics — Model: {MODEL}  ({len(primary_rows)} runs)")
    print("=" * len(header))
    print(header)
    print("-" * len(header))

    for col, label in stat_cols:
        vals = [r[col] for r in primary_rows if col in r]
        if not vals: continue
        s = _stats(vals)
        print(f"{label:<26} {s['mean']:>10} {s['median']:>8} {s['std']:>8} "
              f"{s['min']:>8} {s['max']:>8} {s['P90']:>8} {s['P95']:>8} {s['CV%']:>5}%")

    print("=" * len(header))
    print(f"\nN total rows in ALL_RESULTS: {len(ALL_RESULTS)}  |  primary model rows: {len(primary_rows)}")

Summary Statistics — Model: o4-mini  (25 runs)
Metric                           Mean   Median      Std      Min      Max      P90      P95    CV%
--------------------------------------------------------------------------------------------------
Total latency (s)              58.941   48.361   62.861   31.141  357.077   59.553   66.283 106.7%
  Router (s)                   15.654   15.883    4.594    6.746   24.606   20.304   23.388  29.3%
  Analyze (s)                  24.911   12.916   60.837    4.509  316.121   19.916   20.622 244.2%
  Extract (s)                  15.112   14.847    4.967     8.82   29.757   20.881    23.49  32.9%
  Justify (s)                   3.245    3.171    0.924    1.689    5.985    4.222    4.781  28.5%
  Render (s)                    0.033    0.027    0.014    0.017    0.066    0.052    0.057  41.4%
Total tokens                 14191.56    12596 10928.018     7999    65445  14755.4  16241.4  77.0%
Reasoning tokens               5363.2     5120 1386.256     3

---
## Stage Breakdown Table
Average latency and average token usage per stage, across all primary-model runs.

In [11]:
# ═══════════════════════════════════════════════════════════════════
# Stage Breakdown Table
# Avg latency + avg token usage per stage
# ═══════════════════════════════════════════════════════════════════
primary_rows = [r for r in ALL_RESULTS if r["model"] == MODEL]

if not primary_rows:
    print("No results yet — run experiment cells first.")
else:
    stage_defs = [
        ("router",               "router_latency_s",  "router_prompt_tokens",  "router_completion_tokens",  "router_reasoning_tokens"),
        ("analyze_data",         "analyze_latency_s", "analyze_prompt_tokens", "analyze_completion_tokens", "analyze_reasoning_tokens"),
        ("extract_chart_config", "extract_latency_s", "extract_prompt_tokens", "extract_completion_tokens", "extract_reasoning_tokens"),
        ("justify_chart_type",   "justify_latency_s", "justify_prompt_tokens", "justify_completion_tokens", "justify_reasoning_tokens"),
    ]

    def _avg(rows, col):
        vals = [r[col] for r in rows if col in r]
        return statistics.mean(vals) if vals else 0.0

    total_avg = _avg(primary_rows, "total_latency_s")

    hdr = (f"{'Stage':<24} {'Avg Lat (s)':>12} {'% Total':>8} "
           f"{'Avg Prompt':>11} {'Avg Compl':>10} {'Avg Reason':>11} {'Avg Total Tok':>14}")
    print(f"Stage Breakdown — Model: {MODEL}  ({len(primary_rows)} runs)")
    print("=" * len(hdr))
    print(hdr)
    print("-" * len(hdr))

    llm_lat = 0.0
    for stage, lat_col, pt_col, ct_col, rt_col in stage_defs:
        avg_lat  = _avg(primary_rows, lat_col)
        avg_pt   = _avg(primary_rows, pt_col)
        avg_ct   = _avg(primary_rows, ct_col)
        avg_rt   = _avg(primary_rows, rt_col)
        pct      = avg_lat / total_avg * 100 if total_avg > 0 else 0
        llm_lat += avg_lat
        print(f"{stage:<24} {avg_lat:>11.2f}s {pct:>7.1f}% "
              f"{avg_pt:>10.0f}  {avg_ct:>9.0f}  {avg_rt:>10.0f}  {avg_pt+avg_ct:>13.0f}")

    print("-" * len(hdr))
    avg_render = _avg(primary_rows, "render_latency_s")
    print(f"{'render':<24} {avg_render:>11.2f}s")
    print("-" * len(hdr))
    overhead = total_avg - llm_lat
    print(f"{'framework_overhead':<24} {overhead:>11.2f}s {overhead/total_avg*100:>7.1f}%")
    print(f"{'TOTAL':<24} {total_avg:>11.2f}s {'100.0':>7}%")
    print("=" * len(hdr))

    print(f"\n→ LLM calls account for {llm_lat/total_avg*100:.1f}% of total latency")
    print(f"→ Framework overhead = {overhead/total_avg*100:.1f}% (serialisation, tool dispatch, etc.)")

    # Render success rate
    if RENDER_AVAILABLE:
        n_success = sum(1 for r in primary_rows if r["render_success"])
        print(f"\n→ Render success rate: {n_success}/{len(primary_rows)} "
              f"({n_success/len(primary_rows)*100:.0f}%)")

Stage Breakdown — Model: o4-mini  (25 runs)
Stage                     Avg Lat (s)  % Total  Avg Prompt  Avg Compl  Avg Reason  Avg Total Tok
------------------------------------------------------------------------------------------------
router                         15.65s    26.6%        892       2508        1805           3399
analyze_data                   24.91s    42.3%        746       4090        1480           4836
extract_chart_config           15.11s    25.6%       1842       2463        1761           4305
justify_chart_type              3.24s     5.5%       1258        393         317           1651
------------------------------------------------------------------------------------------------
render                          0.03s
------------------------------------------------------------------------------------------------
framework_overhead              0.02s     0.0%
TOTAL                          58.94s   100.0%

→ LLM calls account for 100.0% of total latency
→ F

In [12]:
# ═══════════════════════════════════════════════════════════════════
# (Optional) JSON export — same data as CSV, structured by experiment
# ═══════════════════════════════════════════════════════════════════
def _mean(lst): return round(statistics.mean(lst), 3) if lst else 0
def _std(lst):  return round(statistics.stdev(lst), 3) if len(lst) > 1 else 0

primary_rows = [r for r in ALL_RESULTS if r["model"] == MODEL]
exp2_r       = [r for r in ALL_RESULTS if r["experiment"] == "exp2_repeated" and r["model"] == MODEL]
exp3_r       = {m: [r for r in ALL_RESULTS if r["experiment"] == "exp3_model_comparison" and r["model"] == m]
                for m in COMPARE_MODELS}
exp4_r       = [r for r in ALL_RESULTS if r["experiment"] == "exp4_prompt_size" and r["model"] == MODEL]
exp5_r       = [r for r in ALL_RESULTS if r["experiment"] == "exp5_variance" and r["model"] == MODEL]

summary = {
    "primary_model":   MODEL,
    "is_reasoning":    MODEL in REASONING_MODELS,
    "total_runs":      len(ALL_RESULTS),
    "exp1_single_run": next((r for r in ALL_RESULTS if r["experiment"] == "exp1_single_run"), {}),
    "exp2_stage_avgs": {
        s: _mean([r[col] for r in exp2_r])
        for s, col in [("router", "router_latency_s"), ("analyze", "analyze_latency_s"),
                       ("extract", "extract_latency_s"), ("justify", "justify_latency_s"),
                       ("total", "total_latency_s")]
    },
    "exp3_model_comparison": {
        m: {"avg_total_s": _mean([r["total_latency_s"] for r in runs]),
            "std_total_s": _std( [r["total_latency_s"] for r in runs]),
            "avg_reasoning_tokens": _mean([r["total_reasoning_tokens"] for r in runs])}
        for m, runs in exp3_r.items() if runs
    },
    "exp4_prompt_scaling": {
        label: {"approx_tokens": exp4[label]["tokens"], "avg_s": round(exp4[label]["avg_s"], 3)}
        for label in exp4
    } if 'exp4' in dir() else {},
    "exp5_variance": {
        "mean_s":  _mean([r["total_latency_s"]       for r in exp5_r]),
        "std_s":   _std( [r["total_latency_s"]       for r in exp5_r]),
        "cv_pct":  round(_std( [r["total_latency_s"] for r in exp5_r]) /
                         (_mean([r["total_latency_s"]for r in exp5_r]) or 1) * 100, 1) if exp5_r else 0,
        "reasoning_tok_min":  min((r["total_reasoning_tokens"] for r in exp5_r), default=0),
        "reasoning_tok_max":  max((r["total_reasoning_tokens"] for r in exp5_r), default=0),
        "reasoning_tok_mean": _mean([r["total_reasoning_tokens"] for r in exp5_r]),
    } if exp5_r else {},
}

json_path = HERE / "bottleneck_thesis_summary.json"
json_path.write_text(json.dumps(summary, indent=2))
print(f"✅ JSON summary saved → {json_path}")
print(json.dumps(summary, indent=2))

✅ JSON summary saved → /Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/FINAL FOLDER/bottleneck_thesis_summary.json
{
  "primary_model": "o4-mini",
  "is_reasoning": true,
  "total_runs": 28,
  "exp1_single_run": {
    "run_number": 1,
    "experiment": "exp1_single_run",
    "model": "o4-mini",
    "total_latency_s": 67.883,
    "router_latency_s": 14.465,
    "analyze_latency_s": 19.946,
    "extract_latency_s": 29.757,
    "justify_latency_s": 3.689,
    "render_latency_s": 0.044,
    "router_prompt_tokens": 974,
    "router_completion_tokens": 2207,
    "router_reasoning_tokens": 1280,
    "analyze_prompt_tokens": 960,
    "analyze_completion_tokens": 3313,
    "analyze_reasoning_tokens": 2816,
    "extract_prompt_tokens": 1922,
    "extract_completion_tokens": 5230,
    "extract_reasoning_tokens": 4416,
    "justify_prompt_tokens": 1425,
    "justify_completion_tokens": 507,
    "justify_reasoning_tokens": 448,
    "total_input_tokens": 5281